In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

%matplotlib inline

# Load the model performance data
df = pd.read_csv(
    "Fraud Detection - ac_thresholds_merged.csv",
    header=0
)
df.columns = ["idx", "score", "TP", "FP", "TN", "FN", "total_check", "Recall", "Precision", "FPR"]
df = df.dropna(subset=["score"]).reset_index(drop=True)
df[["TP","FP","TN","FN"]] = df[["TP","FP","TN","FN"]].astype(float)
df[["Recall","Precision","FPR","score"]] = df[["Recall","Precision","FPR","score"]].astype(float)

# Totals are consistent across all rows (only threshold changes)
total_positives = df["TP"].iloc[0] + df["FN"].iloc[0]   # all fraudsters
total_negatives = df["FP"].iloc[0] + df["TN"].iloc[0]   # all good customers
total = total_positives + total_negatives
prevalence = total_positives / total

# ROC: add boundary points and sort by FPR
fpr_full = np.concatenate([[0.0], df["FPR"].values, [1.0]])
tpr_full = np.concatenate([[0.0], df["Recall"].values, [1.0]])
sort_idx = np.argsort(fpr_full)
fpr_sorted = fpr_full[sort_idx]
tpr_sorted = tpr_full[sort_idx]
auc_score = np.trapz(tpr_sorted, fpr_sorted)

# Precision-Recall: sort by Recall
pr_recall = np.concatenate([[0.0], df["Recall"].values])
pr_precision = np.concatenate([[1.0], df["Precision"].values])
pr_sort = np.argsort(pr_recall)
pr_recall_s = pr_recall[pr_sort]
pr_precision_s = pr_precision[pr_sort]
pr_auc = np.trapz(pr_precision_s, pr_recall_s)

# Lift and flagging rate — computed on df first, then op_df sliced so it inherits them
df["pct_flagged"] = (df["TP"] + df["FP"]) / total * 100
df["lift"] = df["Precision"] / prevalence

# Operating point = first 5 rows (current deployment range)
op_df = df.iloc[:5].copy()


<style>
.reveal h1 { font-size: 1.8em; color: #c0392b; }
.reveal h2 { font-size: 1.4em; color: #c0392b; }
.reveal h3 { font-size: 1.1em; color: #c0392b; }
.reveal { color: #333; }
</style>

# Auth Confidence Model Performance
## Evaluating a binary Classifier with Extreme Class Imbalance and Step Up options


<br><br><br>

<p style="font-size: 0.7em; color: #666;">XGBoost Binary Classifier — Good Customer Score Thresholding</p>


# Presentation Overview

1. **Model Setup** — XGBoost, extreme class imbalance, and the counterfactual problem
2. **ROC AUC** — current Risk Modeling team methodology, full curve, operating point, confusion matrix
3. **Limitations of ROC AUC** — why it misleads under extreme class imbalance
4. **Precision-Recall** — a more informative performance lens
5. **Lift Curve** — adding operational colour to the analysis
6. **Insult Rate** — translating model performance into a business decision


# Part 1: Model Setup — Three Measurement Challenges

## The Model: XGBoost Binary Classifier

- **Goal**: identify fraudulent customers at authentication time
- **Output**: probability of fraud $p_{fraud} \in [0, 1]$
- **Good Customer Score**: $s = 1 - p_{fraud}$ — higher means the customer looks safer
- **Decision rule**: trigger step-up authentication if $s < \theta$

## Three challenges that make performance measurement hard

| # | Challenge | Core issue |
|---|-----------|------------|
| 1 | **Extreme class imbalance** | Fraud prevalence ~0.09% — standard metrics are misleading |
| 2 | **The counterfactual problem** | Abandonment and step-up outcomes are ambiguous — all four confusion matrix cells carry measurement bias |
| 3 | **Timing & attribution** | Multi-step fraud rarely maps to a single auth event — a crude 1-month window over- and under-counts TPs |

The rest of this deck focuses on challenge **#1** (metrics), while keeping **#2** and **#3**
in mind as caveats on the ground-truth reliability of every number shown.


In [ ]:
fig = plt.figure(figsize=(12, 5))

# Donut chart
ax1 = fig.add_subplot(121)
sizes = [total_negatives, total_positives]
labels = [f"Good Customers\n{total_negatives/total*100:.2f}%", f"Fraudsters\n{prevalence*100:.4f}%"]
colors = ["#3498db", "#e74c3c"]
wedges, texts = ax1.pie(sizes, labels=labels, colors=colors, startangle=90,
                         wedgeprops=dict(width=0.55))
texts[0].set_fontsize(12)
texts[1].set_fontsize(12)
ax1.set_title("Dataset Composition", fontsize=13, fontweight="bold")

# Stats table
ax2 = fig.add_subplot(122)
ax2.axis("off")
table_data = [
    ["Total Auth Attempts", f"{total/1e9:.2f} billion"],
    ["Fraudulent Authentication Attempts (positives)", f"{total_positives/1e6:.2f} million"],
    ["Good Customers Authentication Attempts (negatives)", f"{total_negatives/1e9:.2f} billion"],
    ["Prevalence rate", f"{prevalence*100:.4f}%"],
    ["Imbalance ratio", f"1 : {total_negatives/total_positives:,.0f}"],
]
tbl = ax2.table(cellText=table_data, colLabels=["Metric", "Value"],
                loc="center", cellLoc="left")
tbl.auto_set_font_size(False)
tbl.set_fontsize(12)
tbl.scale(1, 2.4)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#c0392b")
        cell.set_text_props(color="white", fontweight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#fdf2f2")
    cell.set_edgecolor("white")

plt.suptitle("Extreme Class Imbalance: The Core Challenge",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## Challenge 1: Measuring Performance — The Confusion Matrix

Every fraud model makes binary decisions at a chosen **score threshold** $\theta$:
flag the customer (predicted fraud) or let them through (predicted legitimate).
Comparing predictions against reality gives four outcomes:

<table style='width:82%; border-collapse:collapse; font-size:0.92em; margin:0 auto;'>
<thead><tr>
  <th style='border:2px solid #ccc; padding:8px;'></th>
  <th style='border:2px solid #ccc; padding:8px; background:#fdecea;'>Actual: Fraudster</th>
  <th style='border:2px solid #ccc; padding:8px; background:#eaf4fb;'>Actual: Legitimate</th>
</tr></thead>
<tbody>
<tr>
  <td style='border:2px solid #ccc; padding:8px; background:#fdecea;'><b>Predicted: Fraud</b><br/><small>(score &lt; &theta;)</small></td>
  <td style='border:2px solid #ccc; padding:8px; background:#d5f5e3; text-align:center;'><b>TP</b> &mdash; True Positive<br/><small>Fraudster correctly flagged</small></td>
  <td style='border:2px solid #ccc; padding:8px; background:#fdecea; text-align:center;'><b>FP</b> &mdash; False Positive<br/><small>Good customer wrongly challenged</small></td>
</tr>
<tr>
  <td style='border:2px solid #ccc; padding:8px; background:#eaf4fb;'><b>Predicted: Legit</b><br/><small>(score &ge; &theta;)</small></td>
  <td style='border:2px solid #ccc; padding:8px; background:#fdecea; text-align:center;'><b>FN</b> &mdash; False Negative<br/><small>Fraudster let through &mdash; missed fraud</small></td>
  <td style='border:2px solid #ccc; padding:8px; background:#d5f5e3; text-align:center;'><b>TN</b> &mdash; True Negative<br/><small>Good customer correctly passed</small></td>
</tr>
</tbody></table>

<br/>

**Prevalence** $= \dfrac{TP + FN}{TP + TN + FP + FN}$ &mdash; the share of true fraudsters in the population (~0.09% here).

Lowering $\theta$ flags more customers: **TP&nbsp;&uarr; and FP&nbsp;&uarr;** simultaneously &mdash; every row of the dataset is one such threshold.


In [ ]:
# Confusion matrix at the first (most conservative) operating-point row
# Each quadrant shows: label / absolute count / % of all records
op = op_df.iloc[0]
TP_v, FP_v = int(op["TP"]), int(op["FP"])
TN_v, FN_v = int(op["TN"]), int(op["FN"])
total_v = TP_v + FP_v + TN_v + FN_v

def fmt(v):
    if v >= 1e9: return f"{v/1e9:.2f}B"
    if v >= 1e6: return f"{v/1e6:.2f}M"
    if v >= 1e3: return f"{v/1e3:.0f}K"
    return str(int(v))

# rows = Predicted (Fraud / Legit), cols = Actual (Fraudster / Legit)
mat  = [[TP_v, FN_v], [FP_v, TN_v]]
bg   = [["#d5f5e3", "#fdecea"], ["#fdecea", "#d5f5e3"]]
lbl  = [["TP", "FN"], ["FP", "TN"]]
desc = [["Fraudster\ncorrectly flagged", "Fraudster\nlet through"],
        ["Good customer\nwrongly challenged", "Good customer\ncorrectly passed"]]

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_xlim(0, 2); ax.set_ylim(0, 2)
ax.set_aspect("equal"); ax.axis("off")

for i in range(2):
    for j in range(2):
        x, y = j, 1 - i
        ax.add_patch(plt.Rectangle((x, y), 1, 1, color=bg[i][j], ec="white", lw=3))
        v = mat[i][j]
        ax.text(x+0.5, y+0.72, lbl[i][j],
                ha="center", va="center", fontsize=20, fontweight="bold", color="#2c3e50")
        ax.text(x+0.5, y+0.47, f"{fmt(v)}  ({v/total_v*100:.3f}%)",
                ha="center", va="center", fontsize=11, color="#2c3e50")
        ax.text(x+0.5, y+0.22, desc[i][j],
                ha="center", va="center", fontsize=8.5, color="#555", style="italic")

ax.text(0.5, 2.12, "Actual: Fraudster", ha="center", va="bottom",
        fontsize=12, fontweight="bold", color="#c0392b")
ax.text(1.5, 2.12, "Actual: Legitimate", ha="center", va="bottom",
        fontsize=12, fontweight="bold", color="#2980b9")
ax.text(-0.08, 1.5, "Predicted:\nFraud", ha="right", va="center",
        fontsize=11, fontweight="bold", color="#c0392b")
ax.text(-0.08, 0.5, "Predicted:\nLegit", ha="right", va="center",
        fontsize=11, fontweight="bold", color="#2980b9")

plt.title(
    f"Confusion Matrix at Score Threshold = {op['score']}"
    f"   (Recall = {op['Recall']:.0%}  |  FPR = {op['FPR']:.2%}  |  Precision = {op['Precision']:.2%})",
    fontsize=12, fontweight="bold", pad=18
)
plt.tight_layout()
plt.show()


## Challenge 2: The Counterfactual Problem

In authentication fraud, **none of the four confusion matrix cells are cleanly observable**.
Every cell carries a measurement bias — and the biases pull in opposite directions.

| What we observe | What it might actually be | Bias direction |
|---|---|---|
| Flagged → **abandons** step-up | Fraudster blocked (TP) *or* legitimate customer who couldn't authenticate — no phone to hand, push notifications off, roaming... | **Overcounts TPs** — recall looks better than it is |
| Flagged → **passes** step-up | Good customer wrongly challenged (FP) *or* sophisticated fraudster who beat the challenge — SIM swap, stolen OTP, social engineering... | **Overcounts FPs, undercounts FNs** — precision looks worse and recall is also understated |
| Not flagged → fraud **never claimed** | True Negative (TN) *but* some fraud goes unreported or absorbed as a write-off | **Undercounts FNs** — model looks better on recall |

### Net effect

These biases partially offset each other but do so **unpredictably** — the balance depends on
step-up UX completion rates and the sophistication mix of the fraud population.

### The mitigation: post-hoc analytics

FP and FN counts are estimated by matching flagged sessions against downstream account behaviour
(transactions, disputes, device signals). This partially resolves the pass/fail ambiguity.
However, **the TP overcount from legitimate abandonment requires separate UX instrumentation**
— distinguishing *"couldn't authenticate"* from *"didn't try"* is a product and data problem,
not a modelling one.


## Challenge 3: The Timing & Attribution Problem

Multi-step fraud rarely maps cleanly onto a single authentication event.
A fraudster may authenticate successfully at time **T₁**, stay dormant, then
commit fraud at time **T₂** — days or weeks later, possibly through a different channel.

### The crude rule: any authentication within 1 month of a fraud claim = fraudulent

This introduces two systematic errors:

| Scenario | Effect | Bias |
|---|---|---|
| Legitimate account holder logs in normally at T₁, account is taken over via a *different* vector at T₂ | The innocent T₁ auth is labelled fraudulent | **Overcounts TPs, undercounts TNs** |
| Fraudster authenticates at T₁, fraud event falls just *outside* the 1-month window | The fraudulent T₁ auth is labelled legitimate | **Undercounts TPs, overcounts TNs** |
| Multiple legitimate authentications occur in the month before a fraud claim | All are labelled fraudulent, inflating the positive class | **Overcounts TPs** — artificially inflates recall |

### The dominant bias

In practice the window tends to **overcount TPs**: any auth near a fraud event gets the
fraudulent label, so the model gets credit for sessions it had no causal relationship with.
This makes recall appear higher than it truly is.

### The path to improvement

A shorter, risk-calibrated window (e.g. 48–72 hours) combined with causal graph analysis
— linking the specific authentication session to the fraud transaction chain — would
significantly reduce attribution noise and give a cleaner ground truth.


# Part 2: Current Methodology — ROC AUC

## What is the ROC Curve?

The **Receiver Operating Characteristic** curve plots, for every possible threshold:

$$\text{TPR (Recall)} = \frac{TP}{TP + FN} \quad\text{vs.}\quad \text{FPR} = \frac{FP}{FP + TN}$$

The **Area Under the Curve (AUC)** summarises discrimination power:
- **AUC = 1.0** → perfect classifier
- **AUC = 0.5** → no-skill baseline (random guessing)

## Current Operating Point

The model is currently deployed at **score thresholds between 0.69 and 0.85**
(first 5 rows of the dataset), catching **22–40% of fraudsters**
while flagging only **1–5% of the good customer base**.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

# Full ROC curve
ax.plot(fpr_sorted, tpr_sorted, color="#c0392b", lw=2.5,
        label=f"XGBoost (AUC = {auc_score:.3f})")

# No-skill diagonal
ax.plot([0, 1], [0, 1], color="#95a5a6", lw=1.5, ls="--", label="No Skill (AUC = 0.500)")

# Shade AUC area
ax.fill_between(fpr_sorted, tpr_sorted, alpha=0.08, color="#c0392b")

# Operating point markers
ax.scatter(op_df["FPR"], op_df["Recall"], color="#e67e22", s=120, zorder=5,
           label="Current operating range (score 0.69–0.85)")
ax.plot(op_df["FPR"], op_df["Recall"], color="#e67e22", lw=2.5)

# AUC annotation
ax.annotate(f"AUC = {auc_score:.3f}",
            xy=(0.45, 0.45), xytext=(0.3, 0.72),
            fontsize=13, fontweight="bold", color="#c0392b",
            arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.5))

# Inset zoom on operating region
axins = ax.inset_axes([0.53, 0.04, 0.44, 0.44])
axins.plot(fpr_sorted, tpr_sorted, color="#c0392b", lw=2)
axins.plot([0, 1], [0, 1], color="#95a5a6", lw=1, ls="--")
axins.scatter(op_df["FPR"], op_df["Recall"], color="#e67e22", s=70, zorder=5)
axins.plot(op_df["FPR"], op_df["Recall"], color="#e67e22", lw=2)
axins.set_xlim(-0.002, 0.07)
axins.set_ylim(0.0, 0.55)
axins.set_title("Operating region (zoomed)", fontsize=9)
axins.grid(True, alpha=0.3)
ax.indicate_inset_zoom(axins, edgecolor="#e67e22")

ax.set_xlabel("False Positive Rate  FP / (FP + TN)", fontsize=12)
ax.set_ylabel("True Positive Rate — Recall  TP / (TP + FN)", fontsize=12)
ax.set_title("ROC Curve — XGBoost Fraud Detection Model", fontsize=14, fontweight="bold")
ax.legend(fontsize=11, loc="lower right")
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrix at the first (most conservative) operating-point row
# Each quadrant shows: label / absolute count / % of all records
op = op_df.iloc[0]
TP_v, FP_v = int(op["TP"]), int(op["FP"])
TN_v, FN_v = int(op["TN"]), int(op["FN"])
total_v = TP_v + FP_v + TN_v + FN_v

def fmt(v):
    if v >= 1e9: return f"{v/1e9:.2f}B"
    if v >= 1e6: return f"{v/1e6:.2f}M"
    if v >= 1e3: return f"{v/1e3:.0f}K"
    return str(int(v))

# rows = Predicted (Fraud / Legit), cols = Actual (Fraudster / Legit)
mat  = [[TP_v, FN_v], [FP_v, TN_v]]
bg   = [["#d5f5e3", "#fdecea"], ["#fdecea", "#d5f5e3"]]
lbl  = [["TP", "FN"], ["FP", "TN"]]
desc = [["Fraudster\ncorrectly flagged", "Fraudster\nlet through"],
        ["Good customer\nwrongly challenged", "Good customer\ncorrectly passed"]]

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_xlim(0, 2); ax.set_ylim(0, 2)
ax.set_aspect("equal"); ax.axis("off")

for i in range(2):
    for j in range(2):
        x, y = j, 1 - i
        ax.add_patch(plt.Rectangle((x, y), 1, 1, color=bg[i][j], ec="white", lw=3))
        v = mat[i][j]
        ax.text(x+0.5, y+0.72, lbl[i][j],
                ha="center", va="center", fontsize=20, fontweight="bold", color="#2c3e50")
        ax.text(x+0.5, y+0.47, f"{fmt(v)}  ({v/total_v*100:.3f}%)",
                ha="center", va="center", fontsize=11, color="#2c3e50")
        ax.text(x+0.5, y+0.22, desc[i][j],
                ha="center", va="center", fontsize=8.5, color="#555", style="italic")

ax.text(0.5, 2.12, "Actual: Fraudster", ha="center", va="bottom",
        fontsize=12, fontweight="bold", color="#c0392b")
ax.text(1.5, 2.12, "Actual: Legitimate", ha="center", va="bottom",
        fontsize=12, fontweight="bold", color="#2980b9")
ax.text(-0.08, 1.5, "Predicted:\nFraud", ha="right", va="center",
        fontsize=11, fontweight="bold", color="#c0392b")
ax.text(-0.08, 0.5, "Predicted:\nLegit", ha="right", va="center",
        fontsize=11, fontweight="bold", color="#2980b9")

plt.title(
    f"Confusion Matrix at Score Threshold = {op['score']}"
    f"   (Recall = {op['Recall']:.0%}  |  FPR = {op['FPR']:.2%}  |  Precision = {op['Precision']:.2%})",
    fontsize=12, fontweight="bold", pad=18
)
plt.tight_layout()
plt.show()


# Part 3: Limitations of ROC AUC

## Why ROC AUC Misleads Under Extreme Imbalance

The ROC curve evaluates the model on **both classes simultaneously**
— and the denominator of the FPR is the count of **all negatives** (good customers).

$$\text{FPR} = \frac{FP}{FP + TN}$$

With **3.6 billion good customers**, even flagging **35 million** (the first operating
point) yields FPR ≈ 0.01 — it looks tiny on the ROC curve.

| What ROC AUC hides | Reality |
|--------------------|---------|
| FPR = 1% looks small | That's **35 million** wrongly flagged good customers |
| AUC = 0.97 looks excellent | Precision at operating point is only **1.9%** |
| No skill line seems far away | The relevant baseline is the **prevalence rate** |

### The Core Problem
A high AUC reflects the model's ability to **rank** fraud vs. non-fraud globally.
It says nothing about **how many good customers get wrongly flagged** per fraud caught
in the region where the model actually operates.


# Part 4: Precision-Recall Analysis

## Why Precision-Recall is Better Here

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN}$$

Precision directly answers: **"Of all the customers we flag, how many are actually fraudsters?"**

- It uses **only positives and true flagged counts** — no TN in the denominator
- The **prevalence line** (horizontal baseline) shows what a random flagging policy achieves
- The further Precision is above the prevalence line, the more value the model adds

| Metric | Good for imbalanced data? | Why |
|--------|--------------------------|-----|
| ROC AUC | Partially | FPR denominator diluted by huge TN count |
| PR AUC | Yes | Precision punishes every false alarm directly |
| Prevalence baseline | Yes | Honest no-skill reference for the rare-class problem |


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# PR curve
ax.plot(pr_recall_s, pr_precision_s, color="#8e44ad", lw=2.5,
        label=f"XGBoost PR curve (AP = {pr_auc:.4f})")

# Prevalence baseline (no-skill horizontal line)
ax.axhline(y=prevalence, color="#95a5a6", lw=1.8, ls="--",
           label=f"No-skill baseline (prevalence = {prevalence*100:.4f}%)")
ax.annotate(f"Prevalence = {prevalence*100:.4f}%\n(random flagging)",
            xy=(0.5, prevalence), xytext=(0.45, prevalence * 25),
            fontsize=10, color="#7f8c8d",
            arrowprops=dict(arrowstyle="->", color="#95a5a6"))

# Shade area under PR curve
ax.fill_between(pr_recall_s, pr_precision_s, prevalence,
                where=(pr_precision_s >= prevalence),
                alpha=0.12, color="#8e44ad", label="Gain over no-skill")

# Operating point
ax.scatter(op_df["Recall"], op_df["Precision"], color="#e67e22", s=120, zorder=5,
           label="Current operating range (score 0.69–0.85)")
ax.plot(op_df["Recall"], op_df["Precision"], color="#e67e22", lw=2.5)

# Annotate operating point precision
first_op = op_df.iloc[0]
ax.annotate(
    f"Operating point\nRecall={first_op['Recall']:.0%}, Prec={first_op['Precision']:.1%}\n"
    f"({first_op['Precision']/prevalence:.0f}x above random)",
    xy=(first_op["Recall"], first_op["Precision"]),
    xytext=(first_op["Recall"] + 0.08, first_op["Precision"] + 0.003),
    fontsize=9, color="#e67e22",
    arrowprops=dict(arrowstyle="->", color="#e67e22")
)

ax.set_xlabel("Recall  TP / (TP + FN)", fontsize=12)
ax.set_ylabel("Precision  TP / (TP + FP)", fontsize=12)
ax.set_title("Precision-Recall Curve — XGBoost Fraud Detection Model",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=10, loc="upper right")
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(0, max(pr_precision_s) * 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


## Limitations of Precision-Recall — Enter the Lift Curve

The PR curve is better than ROC for imbalanced data, but it has its own limitations:

- **Precision is hard to interpret in absolute terms** — 1.9% sounds terrible even when
  it is 20× above the prevalence baseline
- It does not directly communicate the **operational cost**: for every 100 customers
  you flag, how many are real fraudsters?
- The **shape** of the curve is hard to compare across very different prevalence regimes

### The Lift Curve

**Lift** normalises precision against the prevalence rate:

$$\text{Lift} = \frac{\text{Precision}}{\text{Prevalence}} = \frac{P(\text{fraud} \mid \text{flagged})}{P(\text{fraud})}$$

- **Lift = 1** → random flagging, no model value
- **Lift = 20** → flagged customers are **20× more likely** to be fraudsters than average
- **X-axis**: % of the population flagged
- **Y-axis**: how much better than random we are at catching fraud in that group

Lift translates directly to operational language: *"For every 1,000 customers we challenge,
how many real fraudsters do we catch?"*


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: full lift curve ---
ax = axes[0]
ax.plot(df["pct_flagged"], df["lift"], color="#16a085", lw=2.5,
        label="Model Lift")
ax.axhline(y=1.0, color="#95a5a6", lw=1.5, ls="--", label="No-skill (Lift = 1)")

# Operating point
ax.scatter(op_df["pct_flagged"], op_df["lift"], color="#e67e22", s=120, zorder=5,
           label="Current operating range")
ax.plot(op_df["pct_flagged"], op_df["lift"], color="#e67e22", lw=2.5)

ax.set_xlabel("% of Population Flagged", fontsize=12)
ax.set_ylabel("Lift  (Precision / Prevalence)", fontsize=12)
ax.set_title("Lift Curve — Full Population", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# --- Right: zoomed to operating range with annotations ---
ax2 = axes[1]
zoom_df = df[df["pct_flagged"] <= 10]
ax2.plot(zoom_df["pct_flagged"], zoom_df["lift"], color="#16a085", lw=2.5)
ax2.axhline(y=1.0, color="#95a5a6", lw=1.5, ls="--", label="No-skill")
ax2.scatter(op_df["pct_flagged"], op_df["lift"], color="#e67e22", s=120, zorder=5)
ax2.plot(op_df["pct_flagged"], op_df["lift"], color="#e67e22", lw=2.5)

for _, row in op_df.iterrows():
    ax2.annotate(
        f"score={row['score']}\nLift={row['lift']:.0f}x",
        xy=(row["pct_flagged"], row["lift"]),
        xytext=(row["pct_flagged"] + 0.3, row["lift"] + 0.5),
        fontsize=8, color="#e67e22",
        arrowprops=dict(arrowstyle="->", color="#e67e22", lw=0.8)
    )

ax2.set_xlabel("% of Population Flagged", fontsize=12)
ax2.set_ylabel("Lift", fontsize=12)
ax2.set_title("Lift Curve — Operating Region (zoomed)", fontsize=13, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

plt.suptitle(
    "How to read: at Lift=20 we are 20× more efficient at catching fraud than random flagging",
    fontsize=12, style="italic"
)
plt.tight_layout()
plt.show()


# Part 5: The Insult Rate — A Business Decision

## What is the Insult Rate?

The **insult rate** quantifies the price paid in **good-customer friction** for each
fraudster caught. It is the number of **good customers challenged** per **fraudster
stopped**.

$$\text{Insult Rate} = \frac{FP}{TP} = \frac{1 - \text{Precision}}{\text{Precision}}$$

## Why it Matters: Two Outcomes for Flagged Good Customers

When a **legitimate customer** is incorrectly flagged and asked to step up:

| Outcome | Business impact |
|---------|----------------|
| **Passes step-up** | Friction cost only — annoyance, support calls, NPS hit |
| **Abandons** | Lost transaction / customer, revenue impact |

The total cost is therefore:

$$\text{Cost}(\theta) = FP_{pass}(\theta) \times C_{friction} + FP_{abandon}(\theta) \times C_{abandon}$$

To catch a **fixed proportion of fraudsters** (e.g. 40% recall), we must decide how many
good customers we are willing to challenge — whether they succeed or fail.


In [ ]:
df["insult_rate"] = df["FP"] / df["TP"]   # good customers flagged per fraudster caught
df["fp_per_million_caught"] = df["FP"] / df["TP"] * 1e6 / 1e6  # expressed per 1M fraudsters caught

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: Insult rate vs Recall ---
ax = axes[0]
ax.plot(df["Recall"] * 100, df["insult_rate"], color="#2980b9", lw=2.5,
        label="Good customers flagged per fraudster caught")
ax.scatter(op_df["Recall"] * 100, op_df["FP"] / op_df["TP"],
           color="#e67e22", s=120, zorder=5, label="Operating range")
ax.plot(op_df["Recall"] * 100, op_df["FP"] / op_df["TP"],
        color="#e67e22", lw=2.5)

for _, row in op_df.iterrows():
    insult = row["FP"] / row["TP"]
    ax.annotate(
        f"{insult:.0f}:1\n(score {row['score']})",
        xy=(row["Recall"] * 100, insult),
        xytext=(row["Recall"] * 100 + 3, insult + 3),
        fontsize=8, color="#e67e22",
        arrowprops=dict(arrowstyle="->", color="#e67e22", lw=0.8)
    )

ax.set_xlabel("Recall — % of Fraudsters Caught", fontsize=12)
ax.set_ylabel("Good Customers Flagged per Fraudster Caught", fontsize=12)
ax.set_title("Insult Rate vs. Fraud Catch Rate", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# --- Right: absolute FP vs TP at each threshold (operating range detail) ---
ax2 = axes[1]
width = 0.35
x = np.arange(len(op_df))
tp_vals = op_df["TP"].values / 1e6
fp_vals = op_df["FP"].values / 1e6

bars1 = ax2.bar(x - width/2, tp_vals, width, color="#27ae60", alpha=0.8, label="TP (Fraudsters caught) [M]")
bars2 = ax2.bar(x + width/2, fp_vals, width, color="#e74c3c", alpha=0.7, label="FP (Good customers flagged) [M]")

ax2.set_xticks(x)
ax2.set_xticklabels([f"score\n{s:.4f}\nRecall {r:.0%}" 
                     for s, r in zip(op_df["score"], op_df["Recall"])],
                    fontsize=9)
ax2.set_ylabel("Count (millions)", fontsize=12)
ax2.set_title("TP vs FP at Each Operating Threshold", fontsize=13, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis="y")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

plt.suptitle(
    "The Insult Rate Trade-off: catching more fraud means challenging more good customers",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()


## Deciding the Operating Threshold: A Framework

The threshold $\theta$ should be chosen to satisfy a business constraint, not just a
model metric. Three principled approaches:

### 1. Fix the Fraud Catch Rate
"We must catch at least **X%** of fraudsters" (Recall target).
Find the threshold that achieves this at minimum insult rate.

### 2. Fix the Insult Budget
"We can tolerate flagging at most **N** good customers per month"
(absolute FP budget). Pick the highest Recall achievable within that budget.

### 3. Fix the Lift Target
"Every flagged customer must be at least **K times** more likely to be fraud than average"
(Lift ≥ K). This ensures the step-up experience is only triggered for high-risk profiles.

### The Step-Up Abandonment Factor

If **p_abandon** is the probability a good customer abandons when challenged:

$$\text{Effective cost per flagged customer} = p_{abandon} \times C_{lost\ transaction} + (1 - p_{abandon}) \times C_{friction}$$

The insult rate times this cost, weighed against the revenue saved per fraud blocked,
gives the **net expected value of the model at each threshold** — the ultimate decision criterion.


In [ ]:
# Illustrative cost-benefit curve
# Assumptions (illustrative — replace with real business values)
fraud_loss_per_event = 500       # average loss per fraudster not blocked ($)
cost_per_friction    = 2         # cost of step-up for a good customer who passes ($)
p_abandon            = 0.05      # 5% of wrongly challenged good customers abandon
revenue_per_session  = 80        # revenue lost when a good customer abandons ($)

effective_fp_cost = p_abandon * revenue_per_session + (1 - p_abandon) * cost_per_friction

df["fraud_blocked_value"] = df["TP"] * fraud_loss_per_event / 1e6  # $M
df["fp_cost_total"]       = df["FP"] * effective_fp_cost / 1e6     # $M
df["net_value"]           = df["fraud_blocked_value"] - df["fp_cost_total"]

fig, ax = plt.subplots(figsize=(11, 6))

ax.fill_between(df["Recall"] * 100, df["fraud_blocked_value"],
                color="#27ae60", alpha=0.3, label="Value of fraud blocked ($M)")
ax.fill_between(df["Recall"] * 100, df["fp_cost_total"],
                color="#e74c3c", alpha=0.3, label="Cost of good-customer friction ($M)")
ax.plot(df["Recall"] * 100, df["net_value"], color="#2980b9", lw=2.5,
        label="Net Expected Value ($M)")
ax.plot(df["Recall"] * 100, df["fraud_blocked_value"], color="#27ae60", lw=1.5)
ax.plot(df["Recall"] * 100, df["fp_cost_total"], color="#e74c3c", lw=1.5)
ax.axhline(0, color="black", lw=0.8, ls="--")

# Optimal point
opt_idx = df["net_value"].idxmax()
opt_row = df.loc[opt_idx]
ax.axvline(x=opt_row["Recall"] * 100, color="#2980b9", lw=1.5, ls=":",
           label=f"Optimal threshold (Recall={opt_row['Recall']:.0%}, score={opt_row['score']})")
ax.scatter([opt_row["Recall"] * 100], [opt_row["net_value"]],
           color="#2980b9", s=150, zorder=5)

# Operating range
ax.axvspan(op_df["Recall"].min() * 100, op_df["Recall"].max() * 100,
           color="#e67e22", alpha=0.12, label="Current operating range")

ax.set_xlabel("Recall — % of Fraudsters Caught", fontsize=12)
ax.set_ylabel("Value ($M, illustrative)", fontsize=12)
ax.set_title(
    f"Net Expected Value by Threshold\n"
    f"Assumptions: $fraud={fraud_loss_per_event}/event, "
    f"$friction={cost_per_friction}/customer, "
    f"p_abandon={p_abandon:.0%}, "
    f"$revenue_lost={revenue_per_session}/abandon",
    fontsize=12, fontweight="bold"
)
ax.legend(fontsize=10, loc="upper left")
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


# Summary

| Metric | What it tells you | Limitation |
|--------|-------------------|------------|
| **ROC AUC** | Global ranking ability | FPR diluted by huge TN — over-optimistic |
| **Precision-Recall** | Quality of flags at each threshold | Absolute numbers hard to interpret |
| **Lift** | How much better than random | No cost information |
| **Insult Rate** | Operational trade-off | Needs abandon rate to monetise |
| **Net Expected Value** | Business-optimal threshold | Requires cost assumptions |

## Key Takeaways

1. **ROC AUC = 0.97 looks great but hides the FP problem** — at 22% recall we already flag 35M good customers
2. **Precision-Recall + Lift are the right primary metrics** for this class-imbalanced problem
3. **The insult rate frames the business decision**: how many good customers are we willing to challenge to catch each fraudster?
4. **Optimal threshold = business optimum**, not just the best AUC — plug in real cost estimates
5. **Counterfactual FPs** are approximations — invest in the measurement methodology alongside the model
